# 04. 피부 정보 기반 하이브리드 추천

**기존 구성 유지:** KoBERT 콘텐츠 유사도 + SVD 협업 필터링 + 피부 정보 후보 필터.

이번 수정은 결과 하드코딩을 없애고 감성 보정·결측 처리·평가 누수를 수정합니다. 각 단계의 입력·출력과 정책을 아래에 설명합니다.

**입력:** 전처리 CSV, OOF 감성 CSV  
**출력:** 검증된 병합 CSV, 실제 사용자 추천 CSV, 실제 결과 그래프, 추천 평가 리포트.

`content_weight`와 피부 비율 기준은 설정값이며 검증으로 최적화된 값은 아닙니다. 결과는 리뷰 기반 선호도 추천으로, 피부에 대한 의학적 적합성을 보장하지 않습니다.

## 1. 환경과 실행 설정

패키지를 셀 안에서 자동 설치하지 않습니다. 미리 `requirements.txt`를 설치하세요. KoBERT 캐시가 없으면 모델을 다운로드합니다. 모든 사용자 선택과 가중치는 이 셀에서 변경할 수 있습니다.

In [1]:
from pathlib import Path
import sys

# Jupyter 실행 위치에 상관없이 프로젝트 내부에서 루트를 탐색합니다.
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "code/common/project_utils.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("프로젝트 폴더 내부에서 실행하세요.")
common_dir = str(PROJECT_ROOT / "code/common")
if common_dir not in sys.path:
    sys.path.insert(0, common_dir)
from project_utils import SCHEMA_VERSION, add_review_ids, merge_sentiment, known_value, split_concerns

import os
import json
import hashlib
from ast import literal_eval
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from IPython.display import display

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
# 작은 관측 평점 행렬의 CPU 학습에서 과도한 스레드 비용을 줄입니다.
torch.set_num_threads(min(4, os.cpu_count() or 1))
PROCESSED_PATH = PROJECT_ROOT / "outputs/data/processed/Ntoken_review.csv"
SENTIMENT_PATH = PROJECT_ROOT / "outputs/data/sentiment/senti_labeled_df.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs/data/recommendation"
FIGURE_DIR = PROJECT_ROOT / "outputs/figures/recommendation"
REPORT_DIR = PROJECT_ROOT / "outputs/reports/recommendation"
CACHE_DIR = PROJECT_ROOT / "outputs/cache/kobert"
for p in [OUTPUT_DIR, FIGURE_DIR, REPORT_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)
USER_ID = "vwaaang"
TOP_K = 10
CONTENT_WEIGHT = .5
SKIN_THRESHOLD = .1
MODEL_NAME = "monologg/kobert"
# 고정된 모델 버전으로 임베딩 캐시와 결과의 재현성을 확보합니다.
MODEL_REVISION = "38279184ba645e8c94d709fbe92eb5bcb47312c1"
MAX_LENGTH = 128
EMBEDDING_BATCH_SIZE = 16
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print("KoBERT device:", DEVICE)


KoBERT device: mps


## 2. 리뷰 ID로 감성 병합 및 평점 보정

감성 파일을 행 번호로 붙이지 않습니다. ID 집합, 중복, 라벨 정의, 점수 범위를 검사한 뒤 1:1 병합합니다.

`rating_aug = clip(rating + 0.5 × sentiment_score, 1, 5)`

긍정 확률이 0.5보다 높으면 평점을 올리고 낮으면 내립니다. `pred`는 표시용 이진 라벨이며 평점에 직접 더하지 않습니다. 같은 사용자·제품의 여러 리뷰는 협업 학습 전에 평균을 내어 한 쌍으로 만듭니다.

In [2]:
reviews = pd.read_csv(PROCESSED_PATH, encoding="utf-8-sig")
sentiment = pd.read_csv(SENTIMENT_PATH, encoding="utf-8-sig")
merged = merge_sentiment(reviews, sentiment)
merged.to_csv(OUTPUT_DIR / "merged_output.csv", index=False, encoding="utf-8-sig")
df = merged.rename(columns={"customer_name": "user_id", "product_name": "item_id"}).copy()
df["user_id"] = df.user_id.astype(str)
df["item_id"] = df.item_id.astype(str)
df["rating_aug"] = (df.rating + .5 * df.sentiment_score).clip(1, 5)
df["date"] = pd.to_datetime(df.date, errors="coerce")

def join_tokens(value):
    if isinstance(value, list):
        return " ".join(map(str, value))
    if pd.isna(value):
        return ""
    parsed = literal_eval(str(value))
    if not isinstance(parsed, list):
        raise ValueError("Ntoken_review는 리스트 형식이어야 합니다.")
    return " ".join(map(str, parsed))

df["text"] = df.review.fillna("") + " " + df.Ntoken_review.map(join_tokens)
print(f"리뷰 {len(df):,}개 / 사용자 {df.user_id.nunique():,}명 / 제품 {df.item_id.nunique()}개")


리뷰 11,795개 / 사용자 7,985명 / 제품 20개


## 3. KoBERT 리뷰 임베딩

제품 전체 리뷰를 연결해 앞 128토큰만 읽던 방식을 수정했습니다.

1. 리뷰별로 토큰화합니다.
2. 긴 리뷰는 최대 길이 이내의 조각으로 나누어 **전체 토큰을 처리**합니다.
3. 패딩·특수 토큰을 제외한 평균 벡터를 계산합니다.
4. 조각의 유효 토큰 수로 가중 평균하여 리뷰 벡터를 만듭니다.
5. 이후 제품별로 리뷰 벡터를 평균합니다.

캐시는 리뷰 ID·텍스트·모델 버전·임베딩 정책의 해시로 구분합니다. 입력이 달라지면 자동으로 다시 계산합니다. 사전학습 모델은 평가 데이터에 맞춰 추가 학습하지 않습니다.

In [3]:
def masked_mean_pool(hidden, attention_mask, special_tokens_mask):
    mask = (attention_mask.bool() & ~special_tokens_mask.bool()).unsqueeze(-1)
    counts = mask.sum(dim=1).clamp(min=1)
    return (hidden * mask).sum(dim=1) / counts, counts.squeeze(-1)


def encode_reviews(frame):
    config = {"model": MODEL_NAME, "revision": MODEL_REVISION, "max_length": MAX_LENGTH,
              "pooling": "all_chunks_token_weighted_no_special_v1"}
    digest = hashlib.sha256(json.dumps(config, sort_keys=True).encode())
    for row in frame[["review_id", "text"]].itertuples(index=False):
        digest.update(json.dumps(list(row), ensure_ascii=False).encode())
    cache_path = CACHE_DIR / f"{digest.hexdigest()}.npz"
    if cache_path.exists():
        with np.load(cache_path, allow_pickle=False) as cached:
            vectors = cached["vectors"]
            if list(cached["review_ids"]) != frame.review_id.tolist():
                raise ValueError("임베딩 캐시의 리뷰 ID가 다릅니다.")
            if vectors.shape != (len(frame), 768) or not np.isfinite(vectors).all():
                raise ValueError("임베딩 캐시가 손상되었습니다.")
        print("리뷰 임베딩 캐시 사용:", cache_path.name)
        return vectors
    kwargs = {"revision": MODEL_REVISION, "trust_remote_code": True}
    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True, **kwargs)
        model = AutoModel.from_pretrained(MODEL_NAME, local_files_only=True, **kwargs)
    except (OSError, ValueError):
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, **kwargs)
        model = AutoModel.from_pretrained(MODEL_NAME, **kwargs)
    model = model.to(DEVICE).eval()
    payload_size = MAX_LENGTH - tokenizer.num_special_tokens_to_add(pair=False)
    sums = np.zeros((len(frame), model.config.hidden_size), dtype=np.float64)
    token_counts = np.zeros(len(frame), dtype=np.float64)
    pending, owners = [], []

    def flush():
        if not pending:
            return
        enc = tokenizer.pad(pending, padding=True, return_tensors="pt")
        special = enc.pop("special_tokens_mask").to(DEVICE)
        enc = {key: value.to(DEVICE) for key, value in enc.items()}
        with torch.inference_mode():
            hidden = model(**enc).last_hidden_state
            vectors, counts = masked_mean_pool(hidden, enc["attention_mask"], special)
        vectors = vectors.cpu().numpy()
        counts = counts.cpu().numpy()
        for owner, vector, count in zip(owners, vectors, counts):
            sums[owner] += vector * count
            token_counts[owner] += count
        pending.clear()
        owners.clear()

    for i, text in enumerate(frame.text):
        tokens = tokenizer.encode(text, add_special_tokens=False, truncation=False, verbose=False)
        # 빈 리뷰는 전처리에서 제거했지만 빈 토큰 입력도 방어적으로 처리합니다.
        for start in range(0, max(1, len(tokens)), payload_size):
            encoded = tokenizer.prepare_for_model(tokens[start:start+payload_size], add_special_tokens=True,
                                                   return_special_tokens_mask=True, return_attention_mask=True,
                                                   truncation=False, verbose=False)
            pending.append(encoded)
            owners.append(i)
            if len(pending) >= EMBEDDING_BATCH_SIZE:
                flush()
        if (i+1) % 1000 == 0:
            print(f"KoBERT 리뷰 처리: {i+1:,}/{len(frame):,}", flush=True)
    flush()
    vectors = sums / np.maximum(token_counts[:, None], 1)
    vectors /= np.maximum(np.linalg.norm(vectors, axis=1, keepdims=True), 1e-12)
    vectors = vectors.astype(np.float32)
    np.savez_compressed(cache_path, vectors=vectors, review_ids=frame.review_id.to_numpy(dtype=str))
    del model
    if DEVICE.type == "mps":
        torch.mps.empty_cache()
    return vectors


In [4]:
X_review = encode_reviews(df)
print("리뷰 벡터 shape:", X_review.shape)


리뷰 임베딩 캐시 사용: 7f92dda53e219d85ecb5447e9b6000a7cc87c040544e38ec9db5f959d114b51e.npz
리뷰 벡터 shape: (11795, 768)


## 4. 관측 평점만 사용하는 SVD 협업 필터링

Surprise가 설치되어 있으면 기존 SVD를 사용합니다. 없으면 **같은 형태의 편향 포함 잠재요인 모델**을 PyTorch로 학습합니다.

`예상 평점 = 전체 평균 + 사용자 편향 + 제품 편향 + 사용자 벡터 · 제품 벡터`

관측된 사용자·제품 평점의 오차만 최소화하고 정규화를 적용합니다. 미관측 평점을 0으로 채우지 않습니다. 두 구현의 최적화 방식이 달라 결과가 완전히 동일하지는 않으며, 사용한 구현을 리포트에 남깁니다. 신규 사용자는 전체 평균과 제품 편향으로 예측합니다.

In [5]:
class ObservedRatingMF:
    """관측값 기반 SVD 대체 모델. 예측 평점은 1~5로 제한합니다."""
    def __init__(self, n_factors=32, n_epochs=150, learning_rate=.03, regularization=.02, random_state=42):
        self.n_factors = n_factors
        self.n_epochs = n_epochs
        self.learning_rate = learning_rate
        self.regularization = regularization
        self.random_state = random_state

    def fit(self, frame):
        if frame.empty or not frame.rating_aug.between(1, 5).all():
            raise ValueError("유효한 관측 평점이 필요합니다.")
        torch.manual_seed(self.random_state)
        self.users = {v: i for i, v in enumerate(sorted(frame.user_id.astype(str).unique()))}
        self.items = {v: i for i, v in enumerate(sorted(frame.item_id.astype(str).unique()))}
        self.global_mean = float(frame.rating_aug.mean())
        u = torch.tensor(frame.user_id.astype(str).map(self.users).to_numpy(), dtype=torch.long)
        i = torch.tensor(frame.item_id.astype(str).map(self.items).to_numpy(), dtype=torch.long)
        target = torch.tensor(frame.rating_aug.to_numpy(), dtype=torch.float32)
        user_factors = torch.nn.Parameter(torch.randn(len(self.users), self.n_factors) * .05)
        item_factors = torch.nn.Parameter(torch.randn(len(self.items), self.n_factors) * .05)
        user_bias = torch.nn.Parameter(torch.zeros(len(self.users)))
        item_bias = torch.nn.Parameter(torch.zeros(len(self.items)))
        optimizer = torch.optim.Adam([user_factors, item_factors, user_bias, item_bias], lr=self.learning_rate)
        for _ in range(self.n_epochs):
            optimizer.zero_grad()
            prediction = self.global_mean + user_bias[u] + item_bias[i] + (user_factors[u] * item_factors[i]).sum(1)
            # 관측 인덱스에 대해서만 손실/정규화를 계산합니다.
            penalty = (user_factors[u].square().sum(1) + item_factors[i].square().sum(1)
                       + user_bias[u].square() + item_bias[i].square()).mean()
            loss = (prediction - target).square().mean() + self.regularization * penalty
            loss.backward()
            optimizer.step()
        self.user_factors = user_factors.detach().numpy()
        self.item_factors = item_factors.detach().numpy()
        self.user_bias = user_bias.detach().numpy()
        self.item_bias = item_bias.detach().numpy()
        return self

    def predict(self, user_id, item_id):
        u, i = self.users.get(str(user_id)), self.items.get(str(item_id))
        value = self.global_mean
        if u is not None:
            value += float(self.user_bias[u])
        if i is not None:
            value += float(self.item_bias[i])
        if u is not None and i is not None:
            value += float(self.user_factors[u] @ self.item_factors[i])
        return float(np.clip(value, 1, 5))


def fit_collaborative(frame, backend="auto"):
    # 같은 사용자의 동일 제품 반복 리뷰가 학습 가중치를 부풀리지 않게 평균합니다.
    pairs = frame.groupby(["user_id", "item_id"], as_index=False).rating_aug.mean()
    if backend not in {"auto", "surprise", "observed_mf"}:
        raise ValueError("지원하지 않는 협업 모델입니다.")
    if backend in {"auto", "surprise"}:
        try:
            from surprise import Dataset, Reader, SVD
        except ImportError:
            if backend == "surprise":
                raise
        else:
            data = Dataset.load_from_df(pairs[["user_id", "item_id", "rating_aug"]], Reader(rating_scale=(1, 5)))
            model = SVD(n_factors=32, n_epochs=30, random_state=SEED)
            model.fit(data.build_full_trainset())
            return model, "surprise"
    return ObservedRatingMF(random_state=SEED).fit(pairs), "observed_mf"


## 5. 제품 벡터·피부 비율·최신 사용자 정보

제품별 리뷰 벡터를 평균해 콘텐츠 벡터를 만듭니다. 피부 정보 결측은 분모와 조건에서 제외합니다. 피부 고민은 `/`, `,`, `|`로 분리합니다. 고민 비율은 **해당 고민을 가진 리뷰 수 / 고민 정보가 있는 리뷰 수**이므로 여러 고민의 비율 합은 1을 넘을 수 있습니다.

사용자 정보는 날짜순 최신 유효값을 필드별로 사용합니다. 날짜가 없는 리뷰는 알려진 날짜보다 이전에 놓고, 같은 날짜는 리뷰 ID로 순서를 고정합니다.

기존 OR 필터 정책을 유지하되 `Unknown`은 조건으로 사용하지 않습니다. 후보가 없으면 전체 제품으로 완화하고 그 이유를 결과에 표시합니다. 리뷰 비율은 효과 입증이 아니라 사용자 구성 정보입니다.

In [6]:
def get_ratio_pivot(frame, column):
    known = frame.loc[frame[column].map(known_value), ["item_id", "review_id", column]].copy()
    item_ids = sorted(frame.item_id.unique())
    if known.empty:
        return pd.DataFrame(index=item_ids, dtype=float)
    if column == "skin_concerns":
        known[column] = known[column].map(lambda value: sorted(split_concerns(value)))
        known = known.explode(column).dropna(subset=[column])
    denominator = known.groupby("item_id").review_id.nunique()
    counts = known.groupby(["item_id", column]).review_id.nunique().unstack(fill_value=0)
    return counts.div(denominator, axis=0).reindex(item_ids).fillna(0.)


def latest_user_profile(frame, user_id):
    rows = frame.loc[frame.user_id == str(user_id)].sort_values(["date", "review_id"], na_position="first")
    result = {}
    for col in ["skin_type", "skin_tone", "skin_concerns"]:
        known = rows.loc[rows[col].map(known_value), col]
        result[col] = str(known.iloc[-1]) if not known.empty else None
    return result


def build_recommender(frame, review_vectors, backend="auto"):
    if len(frame) != len(review_vectors):
        raise ValueError("리뷰와 벡터 행 수가 다릅니다.")
    items = sorted(frame.item_id.unique())
    index = {item: j for j, item in enumerate(items)}
    codes = frame.item_id.map(index).to_numpy()
    vectors = np.zeros((len(items), review_vectors.shape[1]), dtype=float)
    np.add.at(vectors, codes, review_vectors)
    vectors /= np.bincount(codes, minlength=len(items))[:, None]
    vectors /= np.maximum(np.linalg.norm(vectors, axis=1, keepdims=True), 1e-12)
    model, backend_name = fit_collaborative(frame, backend)
    return {"frame": frame, "items": items, "item_index": index, "vectors": vectors,
            "model": model, "backend": backend_name,
            "ratios": {c: get_ratio_pivot(frame, c) for c in ["skin_type", "skin_tone", "skin_concerns"]}}


def filter_items_by_skin_compatibility(state, user_id, threshold=.1):
    if not 0 <= threshold <= 1:
        raise ValueError("피부 비율 기준은 0~1이어야 합니다.")
    profile = latest_user_profile(state["frame"], user_id)
    conditions, candidates = 0, set()
    for column, value in profile.items():
        if not known_value(value):
            continue
        conditions += 1
        values = split_concerns(value) if column == "skin_concerns" else {value}
        pivot = state["ratios"][column]
        for token in values:
            if token in pivot.columns:
                candidates.update(pivot.index[pivot[token] >= threshold])
    if conditions == 0:
        return set(state["items"]), "no_known_skin_profile"
    if not candidates:
        return set(state["items"]), "skin_filter_relaxed_no_match"
    return candidates, "skin_filter_applied"


## 6. 콘텐츠·협업 점수 결합과 신규 사용자 처리

좋아한 제품(원래 평점 ≥4)의 벡터를 사용자 프로필로 만듭니다. 두 점수는 후보 안에서 0~1 정규화한 뒤 가중 합산합니다. 콘텐츠 프로필이 없으면 협업 점수에 가중치 100%를 줍니다.

신규 사용자는 리뷰 수 기반 인기순을 사용합니다. 동점은 제품명으로 정렬하여 결과를 재현할 수 있습니다. 결과에는 두 원점수·정규화 점수·최종 점수·필터 상태를 함께 표시합니다.

In [7]:
def normalize_scores(scores):
    if not scores:
        return {}
    values = np.array(list(scores.values()), dtype=float)
    if not np.isfinite(values).all():
        raise ValueError("추천 점수에 NaN 또는 무한대가 있습니다.")
    low, high = values.min(), values.max()
    if high - low < 1e-8:
        return {item: .5 for item in scores}
    return {item: float((value-low)/(high-low)) for item, value in scores.items()}


def recommend_content_based(state, user_id):
    frame = state["frame"]
    liked = frame.loc[(frame.user_id == str(user_id)) & (frame.rating >= 4)]
    # 한 제품을 반복 리뷰해도 사용자 프로필에서 과대 가중하지 않습니다.
    weights = liked.groupby("item_id").rating.mean()
    if weights.empty:
        return {}
    indices = [state["item_index"][item] for item in weights.index]
    profile = np.average(state["vectors"][indices], axis=0, weights=weights.to_numpy())
    norm = np.linalg.norm(profile)
    if norm < 1e-12:
        return {}
    scores = state["vectors"] @ (profile/norm)
    return dict(zip(state["items"], map(float, scores)))


def recommend_collaborative_filtering(state, user_id):
    scores = {}
    for item in state["items"]:
        value = state["model"].predict(str(user_id), item)
        scores[item] = float(value.est if state["backend"] == "surprise" else value)
    return scores


RESULT_COLUMNS = ["item_id", "final_score", "content_score", "cf_rating", "content_norm", "cf_norm", "filter_status", "strategy"]

def recommend_hybrid_skin_filtered(state, user_id, k=10, content_weight=.5, skin_filter_threshold=.1):
    if not 0 <= content_weight <= 1 or k < 1:
        raise ValueError("가중치는 0~1, k는 1 이상이어야 합니다.")
    frame = state["frame"]
    candidates, status = filter_items_by_skin_compatibility(state, user_id, skin_filter_threshold)
    seen = set(frame.loc[frame.user_id == str(user_id), "item_id"])
    candidates = sorted(candidates - seen)
    if not candidates:
        return pd.DataFrame(columns=RESULT_COLUMNS)
    if not (frame.user_id == str(user_id)).any():
        # 신규 사용자의 동일 예측값을 임의 순서로 나열하지 않습니다.
        counts = frame.groupby("item_id").user_id.nunique().to_dict()
        scores = normalize_scores({item: counts.get(item, 0) for item in candidates})
        rows = [(item, scores[item], np.nan, np.nan, np.nan, np.nan, status, "cold_start_popularity") for item in candidates]
    else:
        content = recommend_content_based(state, user_id)
        cf = recommend_collaborative_filtering(state, user_id)
        cn = normalize_scores({item: content[item] for item in candidates if item in content})
        fn = normalize_scores({item: cf[item] for item in candidates})
        weight = content_weight if content else 0.
        rows = [(item, weight*cn.get(item, 0.)+(1-weight)*fn[item], content.get(item, np.nan), cf[item],
                 cn.get(item, np.nan), fn[item], status, "hybrid" if content else "collaborative_only") for item in candidates]
    result = pd.DataFrame(rows, columns=RESULT_COLUMNS)
    return result.sort_values(["final_score", "item_id"], ascending=[False, True]).head(k).reset_index(drop=True)


## 7. 실제 사용자 추천 결과 생성

하드코딩된 상품·점수를 제거했습니다. 지정한 사용자가 없으면 명시된 신규 사용자 정책이 적용됩니다. 상위 제품 수는 미리 정해진 10개가 아니라 실제 후보 수에 따라 줄어들 수 있습니다.

In [8]:
state = build_recommender(df, X_review)
recommendations = recommend_hybrid_skin_filtered(state, USER_ID, TOP_K, CONTENT_WEIGHT, SKIN_THRESHOLD)
recommendations.insert(0, "user_id", USER_ID)
recommendations.to_csv(OUTPUT_DIR / "recommendations.csv", index=False, encoding="utf-8-sig")
run_info = {"user_id": USER_ID, "top_k": TOP_K, "content_weight": CONTENT_WEIGHT,
            "skin_filter_threshold": SKIN_THRESHOLD, "cf_backend": state["backend"],
            "model": MODEL_NAME, "model_revision": MODEL_REVISION, "schema_version": SCHEMA_VERSION,
            "review_count": len(df), "item_count": len(state["items"]),
            "profile": latest_user_profile(df, USER_ID)}
(REPORT_DIR / "run_info.json").write_text(json.dumps(run_info, ensure_ascii=False, indent=2), encoding="utf-8")
display(recommendations)
print("협업 모델:", state["backend"])


,user_id,item_id,final_score,content_score,cf_rating,content_norm,cf_norm,filter_status,strategy
0,vwaaang,[대용량기획] 한율 쑥시카 수분크림 110ml (+25ml+ 패드2매+흡착팩폼2ml),0.912437,0.996887,4.955863,0.966768,0.858106,skin_filter_applied,hybrid
1,vwaaang,[9월 올영픽]이즈앤트리 어니언 뉴페어 겔크림 50ml 더블기획 (+20ml),0.874187,0.997710,4.921772,0.999868,0.748506,skin_filter_applied,hybrid
2,vwaaang,[쿨링진정] 아비브 어성초 크림 카밍 튜브 75ml,0.867700,0.997126,4.925047,0.976363,0.759037,skin_filter_applied,hybrid
3,vwaaang,[10.2 하루특가] 라로슈포제 시카플라스트 밤& 크림 2종 기획 (+시카에센스 7...,0.826279,0.995414,4.920706,0.907479,0.745079,skin_filter_applied,hybrid
4,vwaaang,이즈앤트리 어니언 뉴페어 겔크림 80ml 대용량 기획 (+겔크림20ml+패드2매),0.823592,0.991362,4.969761,0.744399,0.902784,skin_filter_applied,hybrid
5,vwaaang,키엘 울트라 훼이셜 오일-프리 젤 크림 50ml,0.818973,0.992816,4.948686,0.802915,0.835030,skin_filter_applied,hybrid
6,vwaaang,[손상/장벽 리페어] 아벤느 시칼파트 플러스 SOS 리페어 크림 100ml 2입 기획,0.814262,0.997713,4.884451,1.000000,0.628524,skin_filter_applied,hybrid
7,vwaaang,마몽드 블루 캐모마일 크림 60ml 기획 (+30ml),0.812128,0.994717,4.920629,0.879424,0.744832,skin_filter_applied,hybrid
8,vwaaang,[응급진정/38% 여드름 개선] 로벡틴 시카케어 블레미쉬 클리어링 크림 50ml 리...,0.810508,0.997130,4.889409,0.976553,0.644463,skin_filter_applied,hybrid
9,vwaaang,[강희재픽/추가증정한정기획] 아벤느 시칼파트 플러스 S.O.S 크림 100ml 2입...,0.805468,0.995962,4.900906,0.929511,0.681425,skin_filter_applied,hybrid


협업 모델: observed_mf


## 8. 피부 타입별 추천과 실제 추천 결과 그래프

새 Figure 3은 건성·지성·복합성 등 **각 타입별 Top 10 추천**입니다. 사용자 최신 피부 타입으로 집단을 구성하고, 집단 내 사용자들의 KoBERT 콘텐츠 유사도와 협업 예측을 각각 평균한 뒤 후보 안에서 정규화하여 결합합니다.

피부 톤·고민의 OR 조건으로 후보를 확장하지 않습니다. 알려진 피부 타입 리뷰 중 해당 타입 비율이 `SKIN_THRESHOLD` 이상인 상품만 남깁니다. 집단 추천에는 구성원이 이미 리뷰한 상품도 포함하며, 후보가 부족해도 다른 타입으로 완화하거나 10개를 억지로 채우지 않습니다. 타입별 점수는 서로 직접 비교할 수 없습니다.

`recommendations_by_skin_type.csv`와 `skin_type_summary.csv`, 통합 그림과 타입별 개별 그림을 저장합니다. 기존 사용자 한 명의 리뷰 구성비 그림과 Figure 4도 별도로 유지합니다.


In [ ]:
import importlib.util
plot_path = PROJECT_ROOT / "code/05_시각화/plot_skin_ratio.py"
spec = importlib.util.spec_from_file_location("recommendation_plots", plot_path)
plots = importlib.util.module_from_spec(spec)
spec.loader.exec_module(plots)
plots.plot_recommendation_figures(df, recommendations, FIGURE_DIR)

from skin_type_recommendations import recommend_by_skin_type
skin_recommendations, skin_summary = recommend_by_skin_type(
    state, k=TOP_K, content_weight=CONTENT_WEIGHT, threshold=SKIN_THRESHOLD)
skin_recommendations.to_csv(OUTPUT_DIR / "recommendations_by_skin_type.csv", index=False, encoding="utf-8-sig")
skin_summary.to_csv(REPORT_DIR / "skin_type_summary.csv", index=False, encoding="utf-8-sig")
plots.plot_skin_type_recommendations(skin_recommendations, skin_summary, FIGURE_DIR)
display(skin_summary)

# 원본 Figure 3과 같은 누적 막대 형식: 타입별 추천 상품의 리뷰 피부 분포
plots.plot_skin_type_review_ratios(df, skin_recommendations, FIGURE_DIR)


## 9. 추천 성능 평가: 사용자별 마지막 제품 보류

평가 가능한 사용자는 서로 다른 제품을 2개 이상 평가했고, 날짜가 알려진 가장 마지막 제품의 평균 평점이 4점 이상인 사용자입니다. 해당 사용자·제품의 **모든 리뷰**를 학습에서 제외합니다. 최신 동점은 제품명으로 고정합니다.

학습 리뷰만으로 제품 벡터·피부 비율·사용자 프로필·협업 모델을 다시 만듭니다. 미리 계산한 리뷰 벡터는 고정 사전학습 모델의 개별 텍스트 변환이므로 재사용합니다.

**감성 OOF 예측은 전체 평점으로 학습된 결과이므로 평가 모델에서는 감성 보정을 끕니다.** 감성까지 포함한 최종 운영 모델의 성능으로 해석하지 마세요. 감성 효과를 검증하려면 외부 분할의 학습 리뷰로 감성 모델까지 재학습해야 합니다.

동일 후보군에서 인기순·평균 평점·협업·콘텐츠·하이브리드를 비교합니다. `Recall@K`, `NDCG@K`, 후보 필터 적용 시 정답 포함률을 저장합니다. 이 분할은 사용자별 보류 평가이며 전역 시간순 미래 예측 실험은 아닙니다.

In [10]:
def make_holdout(frame):
    pairs = frame.groupby(["user_id", "item_id"], as_index=False).agg(rating=("rating", "mean"), date=("date", "max"))
    # 날짜가 누락된 이력이 있는 사용자는 시간순 보류 대상에서 제외합니다.
    eligible_users = frame.groupby("user_id").date.apply(lambda dates: dates.notna().all())
    counts = pairs.groupby("user_id").size()
    eligible = pairs[pairs.user_id.isin(counts[counts >= 2].index) & pairs.user_id.isin(eligible_users[eligible_users].index)]
    last = eligible.sort_values(["date", "item_id"]).groupby("user_id", as_index=False).tail(1)
    heldout = last.loc[last.rating >= 4, ["user_id", "item_id"]].copy()
    keys = set(map(tuple, heldout.to_numpy()))
    train_mask = np.array([(u, i) not in keys for u, i in frame[["user_id", "item_id"]].itertuples(index=False, name=None)])
    return train_mask, heldout


def evaluate_recommendation(frame, vectors, k=10):
    mask, heldout = make_holdout(frame)
    if heldout.empty:
        raise ValueError("평가 가능한 사용자가 없습니다.")
    train = frame.loc[mask].copy()
    train["rating_aug"] = train.rating  # 감성 모델을 통한 평가 정답 누수를 차단합니다.
    evaluation_state = build_recommender(train, vectors[mask])
    popularity = train.groupby("item_id").user_id.nunique().to_dict()
    means = train.groupby(["user_id", "item_id"]).rating.mean().groupby("item_id").mean().to_dict()
    history = {u: set(g.item_id) for u, g in train.groupby("user_id")}
    records = []
    candidate_hits = 0
    for user, target in heldout.itertuples(index=False, name=None):
        candidates, _ = filter_items_by_skin_compatibility(evaluation_state, user, SKIN_THRESHOLD)
        candidates = sorted(candidates - history.get(user, set()))
        candidate_hits += int(target in candidates)
        def rank(scores):
            return sorted(candidates, key=lambda item: (-scores.get(item, 0.), item))[:k]
        cf = recommend_collaborative_filtering(evaluation_state, user)
        content = recommend_content_based(evaluation_state, user)
        rankings = {"popularity": rank(popularity), "mean_rating": rank(means),
                    "collaborative": rank(cf), "content": rank(content),
                    "hybrid": recommend_hybrid_skin_filtered(evaluation_state, user, k, CONTENT_WEIGHT, SKIN_THRESHOLD).item_id.tolist()}
        for method, ranking in rankings.items():
            position = ranking.index(target) + 1 if target in ranking else None
            records.append({"user_id": user, "target_item": target, "method": method,
                            "recall": float(position is not None),
                            "ndcg": 1 / np.log2(position+1) if position else 0.})
    detail = pd.DataFrame(records)
    summary = detail.groupby("method")[["recall", "ndcg"]].mean().rename(columns={"recall": f"Recall@{k}", "ndcg": f"NDCG@{k}"})
    info = {"evaluated_users": len(heldout), "training_reviews": len(train), "heldout_reviews": int((~mask).sum()),
            "candidate_recall": candidate_hits / len(heldout), "cf_backend": evaluation_state["backend"],
            "sentiment_adjustment": False, "protocol": "per-user latest-positive-item holdout, shared skin-filtered candidates",
            "training_items": len(evaluation_state["items"]), "k": k}
    return summary, detail, info


In [11]:
evaluation_summary, evaluation_detail, evaluation_info = evaluate_recommendation(df, X_review, TOP_K)
evaluation_summary.to_csv(REPORT_DIR / "ranking_metrics.csv", encoding="utf-8-sig")
evaluation_detail.to_csv(REPORT_DIR / "ranking_details.csv", index=False, encoding="utf-8-sig")
(REPORT_DIR / "evaluation_info.json").write_text(json.dumps(evaluation_info, ensure_ascii=False, indent=2), encoding="utf-8")
display(evaluation_summary)
print(evaluation_info)


,Recall@10,NDCG@10
method,,
collaborative,0.700557,0.286543
content,0.772981,0.482932
hybrid,0.752089,0.339831
mean_rating,0.515320,0.274042
popularity,0.831476,0.336515


{'evaluated_users': 718, 'training_reviews': 10782, 'heldout_reviews': 1013, 'candidate_recall': 0.9958217270194986, 'cf_backend': 'observed_mf', 'sentiment_adjustment': False, 'protocol': 'per-user latest-positive-item holdout, shared skin-filtered candidates', 'training_items': 20, 'k': 10}
